# Sanity Check: English × TinyAya-Global

Validates the full pipeline on the simplest case before scaling.

**Pass criteria:**
- Probe accuracy > 85% on English harmful/harmless
- Silhouette score > 0 (clusters are separable)
- DiffMean direction aligns with LR direction (cosine > 0.7)

**If this fails:** your hypothesis has problems before you scale to 6 languages × 5 models.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path

from config.models import MODELS
from src.probe import diffmean_probe, logreg_probe, silhouette_score_probe, cosine_alignment

## 1. Load activations (layer 18, 24, 30)

In [ ]:
ACTS_DIR   = Path('../data/processed/activations')
MODEL_KEY  = 'global'
LANG       = 'en'
LAYERS     = MODELS[MODEL_KEY].probe_layers

acts_by_layer = {}
for layer in LAYERS:
    p = ACTS_DIR / MODEL_KEY / LANG / f'layer_{layer}.npy'
    if p.exists():
        acts_by_layer[layer] = np.load(p)
    else:
        print(f'Missing: {p}')

labels = np.load(ACTS_DIR / MODEL_KEY / LANG / 'labels.npy')
print(f'Labels: {labels.sum()} harmful, {(1-labels).sum()} harmless')
for l, a in acts_by_layer.items():
    print(f'  Layer {l}: {a.shape}')

## 2. Probe accuracy and silhouette scores

In [ ]:
results = []

for layer, acts in acts_by_layer.items():
    dm  = diffmean_probe(acts, labels, model_key=MODEL_KEY, lang=LANG, layer=layer)
    lr  = logreg_probe(acts, labels,   model_key=MODEL_KEY, lang=LANG, layer=layer)
    sil = silhouette_score_probe(acts, labels)
    align = cosine_alignment(dm.direction, lr.direction)
    
    results.append({
        'layer':        layer,
        'dm_acc':       dm.accuracy,
        'dm_auc':       dm.auc,
        'lr_acc':       lr.accuracy,
        'lr_auc':       lr.auc,
        'silhouette':   sil,
        'dir_alignment': align,
    })
    print(f'Layer {layer:>3} | DiffMean acc={dm.accuracy:.3f}  '
          f'LR acc={lr.accuracy:.3f}  sil={sil:.3f}  align={align:.3f}')

## 3. Pass/fail summary

In [ ]:
print('\n=== Sanity Check Results ===')
all_pass = True
for r in results:
    lr_ok  = r['lr_acc']  > 0.85
    sil_ok = r['silhouette'] > 0.0
    aln_ok = r['dir_alignment'] > 0.7
    status = 'PASS' if (lr_ok and sil_ok and aln_ok) else 'FAIL'
    if status == 'FAIL': all_pass = False
    print(f"  Layer {r['layer']:>3}: LR={r['lr_acc']:.3f} {'OK' if lr_ok else 'FAIL'}  "
          f"sil={r['silhouette']:.3f} {'OK' if sil_ok else 'FAIL'}  "
          f"align={r['dir_alignment']:.3f} {'OK' if aln_ok else 'FAIL'}  → {status}")

print(f'\nOverall: {"✓ ALL PASS — proceed to full sweep" if all_pass else "✗ FAILURES — investigate before scaling"}')

## 4. PCA visualisation of activation clusters

In [ ]:
from sklearn.decomposition import PCA

fig, axes = plt.subplots(1, len(acts_by_layer), figsize=(5*len(acts_by_layer), 4))
if len(acts_by_layer) == 1:
    axes = [axes]

for ax, (layer, acts) in zip(axes, acts_by_layer.items()):
    pca  = PCA(n_components=2)
    proj = pca.fit_transform(acts)
    colors = ['#2196F3' if l == 0 else '#F44336' for l in labels]
    ax.scatter(proj[:, 0], proj[:, 1], c=colors, alpha=0.5, s=12)
    ax.set_title(f'Layer {layer}  (var={pca.explained_variance_ratio_.sum():.2%})')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

# Legend
from matplotlib.patches import Patch
axes[0].legend(handles=[
    Patch(facecolor='#2196F3', label='harmless'),
    Patch(facecolor='#F44336', label='harmful'),
])
plt.suptitle(f'{MODEL_KEY} / {LANG} — PCA of residual stream', y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/sanity_pca.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Probe accuracy across layers (line plot)

In [ ]:
layers = [r['layer'] for r in results]
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(layers, [r['dm_acc'] for r in results], 'o--', label='DiffMean')
ax.plot(layers, [r['lr_acc'] for r in results], 's-',  label='LogReg')
ax.axhline(0.85, color='gray', linestyle=':', label='85% threshold')
ax.set_xlabel('Layer'); ax.set_ylabel('Accuracy')
ax.set_title(f'Probe accuracy by layer ({MODEL_KEY}/{LANG})')
ax.legend(); ax.set_ylim(0.4, 1.05)
plt.tight_layout()
plt.savefig('../results/figures/sanity_probe_acc.png', dpi=150, bbox_inches='tight')
plt.show()